In [17]:
import os
import h5py
import numpy as np
import pandas as pd

#Define paths relative to repo root

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
raw_dir = os.path.join(repo_root, "data", "raw")
cleaned_dir = os.path.join(repo_root, "data", "cleaned")
os.makedirs(cleaned_dir, exist_ok=True)

raw_meta_path = os.path.join(raw_dir, "metadata.csv")
raw_wave_path = os.path.join(raw_dir, "waveforms.hdf5")

# 1. Load raw metadata
print("Loading raw metadata...")
df = pd.read_csv(raw_meta_path)
initial_count = len(df)
print(f"Initial record count: {initial_count}")

# Track cleaening logs for A3
cleaning_log = {}

# 2. A2: Homogenization checks
print("\n--- A2: Homogenization ---")

# Ensure expected column types and handle coordinate/time consistency
if "trace_sampling_rate_hz" in df.columns:
    unique_rates = df["trace_sampling_rate_hz"].unique()
    print(f"Sampling rates present (Hz): {unique_rates}")

# 3. A3: Quality and gap cleaning rules
print("\n--- A3: Applying physical cleaning rules ---")



Loading raw metadata...
Initial record count: 18640

--- A2: Homogenization ---
Sampling rates present (Hz): [100]

--- A3: Applying physical cleaning rules ---


In [18]:
# Rule #1: Drop rows missing key seismic metadata ( magnitude, station coordinates)
req_cols = ["trace_name", "preferred_source_magnitude", "station_latitude_deg", "station_longitude_deg"] 
df_clean = df.dropna(subset=req_cols).copy()
dropped_missing_meta = initial_count - len(df_clean)
cleaning_log["dropped_missing_meta"] = dropped_missing_meta
print(f"Rule 1 - Dropped due to missing key metadata: {dropped_missing_meta}")



Rule 1 - Dropped due to missing key metadata: 0


In [19]:
# Rule #2: Physical magnitude bounds check (filter out unrealistic magnitudes)
if "preferred_source_magnitude" in df_clean.columns:
    before_mag = len(df_clean)
    df_clean = df_clean[(df_clean["preferred_source_magnitude"] >= 0.0) & (df_clean["preferred_source_magnitude"] <= 9.5)]
    dropped_mag = before_mag - len(df_clean)
    cleaning_log["dropped_unrealistic_magnitude"] = dropped_mag
    print(f"Rule 2 - Dropped due to out of bounds magnitude: {dropped_mag}")



Rule 2 - Dropped due to out of bounds magnitude: 0


In [20]:
import os
import h5py
import numpy as np
import pandas as pd

# Define paths relative to repository root
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
raw_dir = os.path.join(repo_root, "data", "raw")
clean_dir = os.path.join(repo_root, "data", "clean")
os.makedirs(clean_dir, exist_ok=True)

raw_meta_path = os.path.join(raw_dir, "metadata.csv")
raw_wave_path = os.path.join(raw_dir, "waveforms.hdf5")

# 1. Load Raw Metadata
print("Loading raw metadata...")
df = pd.read_csv(raw_meta_path)
initial_count = len(df)
print(f"Initial record count: {initial_count}")

# 2. Re-apply Rules 1 & 2
req_cols = ["trace_name", "preferred_source_magnitude", "station_latitude_deg", "station_longitude_deg", "bucket", "tindex"]
df_clean = df.dropna(subset=req_cols).copy()

if "preferred_source_magnitude" in df_clean.columns:
    df_clean = df_clean[(df_clean["preferred_source_magnitude"] >= 0.0) & (df_clean["preferred_source_magnitude"] <= 9.5)].copy()

# 3. Rule 3: Direct HDF5 Bucket Array Indexing via tindex
print("\nVerifying waveform trace quality in HDF5 bucket datasets...")
valid_indices = []
invalid_trace_count = 0

with h5py.File(raw_wave_path, "r") as hf:
    data_group = hf["data"]
    
    for idx, row in df_clean.iterrows():
        b_val = str(row["bucket"]).strip()
        t_idx = int(row["tindex"])
        
        # Ensure bucket prefix matches HDF5 key format ('bucket1', 'bucket2', etc.)
        bucket_key = b_val if b_val.startswith("bucket") else f"bucket{b_val}"
        
        if bucket_key in data_group:
            bucket_dataset = data_group[bucket_key]
            if t_idx < bucket_dataset.shape[0]:
                data = bucket_dataset[t_idx]
                
                # Check for NaNs, Infs, or dead (all-zero) channels
                if not np.isnan(data).any() and not np.isinf(data).any() and np.abs(data).max() > 0:
                    valid_indices.append(idx)
                else:
                    invalid_trace_count += 1
            else:
                invalid_trace_count += 1
        else:
            invalid_trace_count += 1

df_clean = df_clean.loc[valid_indices].copy()
print(f"Rule 3 - Dropped due to NaN/flatline/missing waveforms: {invalid_trace_count}")
print(f"Final clean dataset size: {len(df_clean)} records (Retained {len(df_clean)/initial_count*100:.2f}%)")

# 4. Save Clean Artifacts (A4)
clean_meta_parquet = os.path.join(clean_dir, "metadata_clean.parquet")
clean_meta_csv = os.path.join(clean_dir, "metadata_clean.csv")

df_clean.to_parquet(clean_meta_parquet, index=False)
df_clean.to_csv(clean_meta_csv, index=False)
print(f"\nSaved clean metadata to: {clean_meta_parquet}")

cleaning_summary_path = os.path.join(clean_dir, "DATASET_SUMMARY.md")
with open(cleaning_summary_path, "w") as f:
    f.write("# Dataset Cleaning & Quality Summary (A2-A4)\n\n")
    f.write(f"- **Initial Raw Records:** {initial_count}\n")
    f.write(f"- **Final Clean Records:** {len(df_clean)}\n")
    f.write(f"- **Dropped corrupted/flatlined waveforms:** {invalid_trace_count}\n\n")
    f.write("## Data Format (A4):\n")
    f.write("- Clean metadata stored in `data/clean/metadata_clean.parquet`\n")
    f.write("- Waveforms indexed by `bucket` and `tindex` into `data/raw/waveforms.hdf5`.\n")

print(f"Saved summary to: {cleaning_summary_path}")

Loading raw metadata...
Initial record count: 18640



Verifying waveform trace quality in HDF5 bucket datasets...
Rule 3 - Dropped due to NaN/flatline/missing waveforms: 0
Final clean dataset size: 18640 records (Retained 100.00%)

Saved clean metadata to: /home/mirandastrap/projects/MLGEO_AU26_seismology/data/clean/metadata_clean.parquet
Saved summary to: /home/mirandastrap/projects/MLGEO_AU26_seismology/data/clean/DATASET_SUMMARY.md
